# Week 13 — Python Solution Lab
## Waves and Sound

**Companion to `notebooks/Week_13.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_13.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P1` | Reading a Travelling Wave Equation | crest tracking to *measure* wave speed |
| **L2 · Intermediate** | `P6` | Open and Closed Organ Pipes | harmonic series, missing even harmonics |
| **L3 · Challenge** | `P10` | Fourier Decomposition of a Plucked String | **Fourier synthesis** + projection integrals |

---

## L1 · Basic — P1: Reading a Travelling Wave Equation

> **Problem (Week_13.ipynb, L1 — P1).** A transverse wave on a string is
> $y(x,t) = 0.05\sin(3.0x - 12t)$ (SI units). Find the amplitude, wavelength, frequency, and
> wave speed.

**Diagram → Principle.** Match the given form against the standard
$y = A\sin(kx - \omega t)$ and read off the parameters. The minus sign means it travels in $+x$.

**Equation.** $\lambda = 2\pi/k$, $f = \omega/2\pi$, $v = \omega/k = f\lambda$.

**Hand prediction.** $A = 0.05$ m, $\lambda = 2.094$ m, $f = 1.91$ Hz, $v = 4.0$ m/s.

**What Python adds.** We *demonstrate* the wave speed rather than asserting it: track a single
crest across successive snapshots and measure how far it moves per unit time. Getting $4.0$ m/s
out of the animation frames — independently of the $\omega/k$ formula — is what makes "wave speed"
mean something physical instead of being a ratio of two symbols.

In [ ]:
# ═══ W13 · L1 · P1 — Reading a wave equation, then MEASURING the speed ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL: y(x,t) = A sin(k x - w t) -----------------------------------
A, k_wave, w = 0.05, 3.0, 12.0
y = lambda x, t: A*np.sin(k_wave*x - w*t)

# --- PREDICT: read the parameters off the equation ----------------------
lam  = 2*np.pi/k_wave
f    = w/(2*np.pi)
T    = 1/f
v    = w/k_wave
print(f"amplitude   A      = {A:.4f} m")
print(f"wave number k      = {k_wave:.4f} rad/m  ->  lambda = 2pi/k = {lam:.4f} m")
print(f"angular freq omega = {w:.4f} rad/s  ->  f = w/2pi = {f:.4f} Hz  (T = {T:.4f} s)")
print(f"wave speed  v = w/k = {v:.4f} m/s")
print(f"cross-check v = f * lambda = {f*lam:.4f} m/s  -> agrees")
assert np.isclose(v, f*lam)
print("the sign is (kx - wt), so the wave travels in the +x direction.")

# --- VERIFY by TRACKING A CREST across snapshots ------------------------
xs = np.linspace(0, 4*lam, 400001)
times = np.linspace(0, T, 9)
crest_x = []
for t in times:
    # the first crest at or after x = 0
    idx = np.argmax(y(xs, t) > A*0.999999)
    crest_x.append(xs[idx])
crest_x = np.array(crest_x)

# unwrap: the tracked crest jumps by lambda when it leaves the window
dx = np.diff(crest_x)
dx[dx < -lam/2] += lam
v_measured = np.mean(dx/np.diff(times))
print(f"\nMEASURED from crest tracking: v = {v_measured:.4f} m/s "
      f"(formula said {v:.4f})")
assert abs(v_measured - v) < 0.01

print(f"\n  in one full period T = {T:.4f} s the pattern advances exactly one wavelength:")
print(f"    v * T = {v*T:.4f} m   vs   lambda = {lam:.4f} m")
assert np.isclose(v*T, lam)

# --- Plot: snapshots and a single-point time trace ----------------------
xs_plot = np.linspace(0, 2*lam, 800)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for i, t in enumerate(np.linspace(0, T/2, 5)):
    ax1.plot(xs_plot, y(xs_plot, t), lw=2, alpha=0.35 + 0.65*i/4,
             color="#1565c0", label=f"t = {t:.3f} s" if i in (0, 4) else None)
ax1.annotate("", xy=(lam*0.75, 0.062), xytext=(lam*0.25, 0.062),
             arrowprops=dict(arrowstyle="->", color="#e65100", lw=2))
ax1.text(lam*0.5, 0.066, "travel", color="#e65100", ha="center", fontsize=10)
ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)"); ax1.set_ylim(-0.08, 0.08)
ax1.set_title(f"snapshots in space ($\\lambda$ = {lam:.3f} m)")
ax1.grid(alpha=.3); ax1.legend(fontsize=8)

ts = np.linspace(0, 2*T, 600)
ax2.plot(ts, y(0.0, ts), color="#2e7d32", lw=2)
ax2.set_xlabel("t (s)"); ax2.set_ylabel("y (m)")
ax2.set_title(f"one point (x = 0) in time (T = {T:.3f} s)")
ax2.grid(alpha=.3)
plt.suptitle("W13 P1 — $y = 0.05\\sin(3x - 12t)$", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(lam - 2.0944) < 1e-3 and abs(f - 1.9099) < 1e-3 and abs(v - 4.0) < 1e-9
print(f"[OK] Matches textbook answer: A = {A} m, lambda = {lam:.3f} m, "
      f"f = {f:.2f} Hz, v = {v:.1f} m/s")

## L2 · Intermediate — P6: Open and Closed Organ Pipes

> **Problem (Week_13.ipynb, L2 — P6).** An organ pipe open at both ends has a fundamental of
> $256$ Hz ($v = 343$ m/s). (a) Pipe length? (b) Fundamental of a same-length pipe closed at one
> end? (c) First three resonant frequencies of each.

**Diagram → Principle.** Boundary conditions decide everything. Open ends force displacement
antinodes; a closed end forces a node. An open–open pipe fits $\lambda/2$; an open–closed pipe
fits only $\lambda/4$, so it sounds an **octave lower** — and can only produce odd harmonics.

**Equation.** Open: $f_n = nv/2L$, $n = 1,2,3\ldots$; closed: $f_n = nv/4L$, $n = 1,3,5\ldots$.

**Hand prediction.** $L = 343/(2\times256) = 0.670$ m; closed fundamental $= 128$ Hz.

**What Python adds.** Generating both harmonic series programmatically makes the **missing even
harmonics** of the closed pipe impossible to overlook — that absence is exactly why a stopped pipe
sounds hollow rather than merely lower. We also convert the frequency ratios to musical intervals
in cents, connecting the physics to what a listener actually hears.

In [ ]:
# ═══ W13 · L2 · P6 — Boundary conditions decide the harmonic series ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
v, f1_open = 343.0, 256.0

# --- (a) length from the open-open fundamental --------------------------
L = v / (2*f1_open)
print(f"(a) open-open pipe fits lambda/2, so L = v/(2 f1) = {L:.4f} m")

# --- (b) the same tube stopped at one end -------------------------------
f1_closed = v / (4*L)
print(f"(b) closed at one end it fits lambda/4, so f1 = v/(4L) = {f1_closed:.2f} Hz")
print(f"    ratio to the open pipe = {f1_open/f1_closed:.1f}  -> exactly one octave LOWER")
assert np.isclose(f1_open/f1_closed, 2.0)

# --- (c) the two harmonic series ----------------------------------------
n_open   = np.arange(1, 7)                 # all integers
n_closed = np.arange(1, 12, 2)             # ODD integers only
f_open   = n_open   * v/(2*L)
f_closed = n_closed * v/(4*L)

print(f"\n(c) open-open pipe  (all harmonics):")
for n, f in zip(n_open, f_open):
    print(f"      n = {n}: {f:8.2f} Hz")
print(f"    open-closed pipe (ODD harmonics only):")
for n, f in zip(n_closed, f_closed):
    print(f"      n = {n}: {f:8.2f} Hz")

# --- The missing harmonics, stated explicitly ---------------------------
missing = [n*v/(4*L) for n in (2, 4, 6)]
print(f"\n  The closed pipe CANNOT produce {', '.join(f'{f:.0f} Hz' for f in missing)}.")
print("  A closed end must be a displacement node, which forbids the even harmonics.")
print("  That missing-even-harmonic spectrum is why a stopped pipe sounds hollow,")
print("  not merely lower in pitch.")
assert all(not np.any(np.isclose(f_closed, mf)) for mf in missing)

# --- Musical interpretation ---------------------------------------------
cents = lambda a, b: 1200*np.log2(a/b)
print(f"\n  intervals above each fundamental (cents; 1200 = one octave):")
print(f"    open   n=2: {cents(f_open[1], f_open[0]):7.1f}  n=3: {cents(f_open[2], f_open[0]):7.1f}")
print(f"    closed n=3: {cents(f_closed[1], f_closed[0]):7.1f}  n=5: {cents(f_closed[2], f_closed[0]):7.1f}")
print("    the closed pipe's second available tone is a twelfth (1902 c), not an octave.")

# --- Plot ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.vlines(f_open, 0, 1, color="#1565c0", lw=3, label="open-open")
ax1.vlines(f_closed, 0, 0.8, color="#e65100", lw=3, label="open-closed")
ax1.vlines(missing, 0, 0.8, color="crimson", lw=2, ls=":", label="forbidden in closed pipe")
ax1.set_xlabel("frequency (Hz)"); ax1.set_yticks([])
ax1.set_title("harmonic content"); ax1.legend(fontsize=8)

xs = np.linspace(0, L, 500)
for i, (n, off, col) in enumerate([(1, 0, "#1565c0"), (2, -2.6, "#1565c0"), (3, -5.2, "#1565c0")]):
    ax2.plot(xs, off + np.cos(n*np.pi*xs/L), color=col, lw=2)
    ax2.plot(xs, off - np.cos(n*np.pi*xs/L), color=col, lw=2, alpha=.4)
    ax2.text(L*1.02, off, f"n={n}", fontsize=9, va="center")
ax2.set_xlabel("position along the open-open pipe (m)"); ax2.set_yticks([])
ax2.set_title("displacement antinodes at both open ends")
for ax in (ax1, ax2): ax.grid(alpha=.3, axis="x")
plt.suptitle("W13 P6 — the same tube, two boundary conditions", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(L - 0.6699) < 1e-3 and abs(f1_closed - 128.0) < 0.01
assert np.allclose(f_open[:3], [256.0, 512.0, 768.0])
assert np.allclose(f_closed[:3], [128.0, 384.0, 640.0])
print(f"[OK] Matches textbook answer: L = {L:.3f} m; closed f1 = {f1_closed:.0f} Hz;")
print(f"     open 256/512/768 Hz, closed 128/384/640 Hz.")

## L3 · Challenge — P10: Fourier Decomposition of a Plucked String

> **Problem (Week_13.ipynb, L3 — P10).** A string of length $L = 2.0$ m, linear density
> $\mu = 0.010$ kg/m and tension $T = 40$ N is plucked so that its initial shape is a triangle
> with peak displacement $h = 0.02$ m at $x = L/3$. **(a)** Find the fundamental frequency and
> wave speed. **(b)** The Fourier coefficient of the $n$-th harmonic is stated as
> $b_n = \dfrac{9h}{2n^2\pi^2}\sin\!\left(\dfrac{n\pi}{3}\right)$; calculate $b_1$, $b_2$, $b_3$.
> **(c)** Which harmonics are absent, and why?

**Diagram → Principle.** A plucked shape is not a single mode. It is a **superposition** of all
the string's standing modes, each with its own amplitude $b_n$, and each then evolving at its own
frequency $f_n = nf_1$.

**Equation.** $v = \sqrt{T/\mu}$, $f_1 = v/2L$; $y(x,0) = \sum_n b_n \sin(n\pi x/L)$.

**Hand prediction.** $v = \sqrt{40/0.01} = 63.25$ m/s, $f_1 = 15.81$ Hz.

> ⚠️ **The $b_n$ formula printed in the problem is off by a factor of 2.** The projection
> integral $b_n = \frac{2}{L}\int_0^L y(x,0)\sin\frac{n\pi x}{L}\,dx$ — which *defines* the
> coefficients — gives
> $$b_n = \frac{2hL^2}{n^2\pi^2\,x_p(L-x_p)}\sin\frac{n\pi x_p}{L} \;\xrightarrow{\;x_p=L/3\;}\; \frac{9h}{n^2\pi^2}\sin\frac{n\pi}{3},$$
> i.e. **without** the $2$ in the denominator. So $b_1 = 15.79$ mm, not $7.90$ mm. The cell below
> computes the integral numerically for each $n$ and confirms the corrected formula exactly; the
> reconstruction converging onto the true triangle is further proof (with the halved coefficients
> it would converge to a triangle of half the height).

**What Python adds.** Everything. We **reconstruct** the triangular pluck by summing the series
and watch it converge as more harmonics are included — the single most convincing demonstration
of Fourier synthesis available in a first-year course. Because we check $b_n$ against the defining
integral rather than trusting the printed formula, the factor-of-2 error surfaces immediately. We
also show that every third harmonic vanishes because the pluck point at $L/3$ is a node of those
modes, and plot the string's subsequent motion.

In [ ]:
# ═══ W13 · L3 · P10 — Fourier synthesis of a plucked string ═══
import numpy as np
from scipy.integrate import quad
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
L, mu, T_ten, h, xp = 2.0, 0.010, 40.0, 0.02, 2.0/3.0     # pluck at x = L/3

# --- (a) wave speed and fundamental -------------------------------------
v  = np.sqrt(T_ten/mu)
f1 = v/(2*L)
print(f"(a) v  = sqrt(T/mu) = {v:.4f} m/s")
print(f"    f1 = v/(2L)     = {f1:.4f} Hz   (T1 = {1/f1:.5f} s)")

# --- The initial shape: a triangle peaking at x = L/3 -------------------
def pluck(x):
    x = np.asarray(x, dtype=float)
    return np.where(x <= xp, h*x/xp, h*(L - x)/(L - xp))

# --- (b) the given coefficients, and an independent check ---------------
def b_printed(n):
    """The formula as printed in the problem statement -- half the correct value."""
    return 9*h/(2*n**2*np.pi**2) * np.sin(n*np.pi/3)

def b_given(n):
    """CORRECTED closed form for a triangular pluck at x_p = L/3."""
    return 9*h/(n**2*np.pi**2) * np.sin(n*np.pi/3)

def b_general(n, xp=xp):
    """General triangular pluck of peak h at x_p: valid for any pluck point."""
    return 2*h*L**2/(n**2*np.pi**2*xp*(L - xp)) * np.sin(n*np.pi*xp/L)

def b_numeric(n):
    """Projection integral b_n = (2/L) int_0^L y(x) sin(n pi x / L) dx -- the DEFINITION."""
    f = lambda x: pluck(x)*np.sin(n*np.pi*x/L)
    val, _ = quad(f, 0, L, points=[xp], limit=200)
    return 2/L*val

# The printed formula fails the definition by exactly a factor of 2:
print(f"\n  FORMULA CHECK at n = 1:")
print(f"    projection integral (definition) : {b_numeric(1)*1000:9.5f} mm")
print(f"    formula as printed in the problem: {b_printed(1)*1000:9.5f} mm  <- half")
print(f"    corrected closed form            : {b_given(1)*1000:9.5f} mm")
print(f"    ratio integral/printed = {b_numeric(1)/b_printed(1):.4f}")
assert abs(b_numeric(1)/b_printed(1) - 2.0) < 1e-6

print(f"\n(b) harmonic content (b_n in mm, f_n = n * f1):")
ns = np.arange(1, 13)
bg = np.array([b_given(n) for n in ns])
bn = np.array([b_numeric(n) for n in ns])

# Modal energy E_n is proportional to (n b_n)^2. Normalise against the TOTAL
# over all modes, not just the 12 shown -- otherwise the percentages are a
# share of a truncation, which flatters the low harmonics.
E_shown = (bg*ns)**2
N_ALL   = 20001
ns_all  = np.arange(1, N_ALL)
E_total = float((( np.array([b_given(n) for n in ns_all]) * ns_all )**2).sum())
energy  = 100*E_shown/E_total
print(f"    (energy % is each mode's share of the TOTAL over all modes, "
      f"not just these 12)")
print(f"  {'n':>3s} {'f_n (Hz)':>10s} {'b_n given':>12s} {'b_n numeric':>13s} {'energy %':>10s}")
for n, g_, nm_, e in zip(ns, bg, bn, energy):
    note = "   <- SILENT (node at the pluck point)" if abs(g_) < 1e-15 else ""
    print(f"  {n:3d} {n*f1:10.2f} {g_*1000:12.5f} {nm_*1000:13.5f} {e:10.2f}{note}")
    assert abs(g_ - nm_) < 1e-9, f"coefficient mismatch at n = {n}"
    assert abs(g_ - b_general(n)) < 1e-12, f"general formula mismatch at n = {n}"
print("  the CORRECTED formula matches the projection integral for every n. [verified]")
print("  (the general form also reproduces it, so it is not a coincidence of x_p = L/3)")

# --- the energy normalisation, made explicit ----------------------------
print(f"\n  energy bookkeeping:")
print(f"    fundamental holds {energy[0]:.2f}% of the TOTAL modal energy")
print(f"    the 12 modes listed hold {energy.sum():.2f}% between them")
print(f"    had we normalised to these 12 alone, n=1 would read "
      f"{100*E_shown[0]/E_shown.sum():.2f}% -- flattering, and wrong")
assert abs(energy[0] - 68.39) < 0.05, f"fundamental share = {energy[0]}"
assert 96.0 < energy.sum() < 96.5

# --- (b) the three coefficients the problem actually asks for -----------
print(f"\n(b) answers as asked, using the CORRECTED formula:")
for n in (1, 2, 3):
    print(f"    b_{n} = {b_given(n)*1000:8.4f} mm    "
          f"(printed formula would give {b_printed(n)*1000:8.4f} mm)")

# --- (c) which harmonics are absent, and why ----------------------------
absent = [int(n) for n in ns if abs(b_given(n)) < 1e-15]
print(f"\n(c) absent harmonics among n = 1..12: {absent}")
print(f"    sin(n*pi/3) = 0 exactly when 3 divides n. Physically, the pluck point")
print(f"    x_p = L/3 is a NODE of modes 3, 6, 9, ... -- you cannot excite a mode by")
print(f"    displacing it at a point where that mode has no displacement.")
assert absent == [3, 6, 9, 12]

# --- Why every third harmonic is missing --------------------------------
print(f"\n  n = 3, 6, 9, ... vanish because sin(n pi/3) = 0 for n divisible by 3.")
print(f"  Physically: the pluck point x = L/3 is a NODE of those modes, so plucking")
print(f"  there cannot excite them. Move the pluck and they reappear:")
for xp_alt in (L/2, L/4):
    n3 = 2/L*quad(lambda x: np.where(x <= xp_alt, h*x/xp_alt, h*(L-x)/(L-xp_alt))
                            * np.sin(3*np.pi*x/L), 0, L, points=[xp_alt])[0]
    print(f"    plucked at x = {xp_alt:.3f} m -> b_3 = {n3*1000:+.5f} mm")

# --- Reconstruct the pluck and watch it converge ------------------------
xs = np.linspace(0, L, 1200)
def synth(N):
    return sum(b_given(n)*np.sin(n*np.pi*xs/L) for n in range(1, N+1))

print(f"\n  convergence of the reconstruction (RMS error vs the true triangle):")
for N in (1, 2, 5, 10, 25, 100, 400):
    err = np.sqrt(np.mean((synth(N) - pluck(xs))**2))
    print(f"    {N:4d} harmonics -> RMS error {err*1000:8.5f} mm")
assert np.sqrt(np.mean((synth(400) - pluck(xs))**2)) < 1e-5

# --- Plot: synthesis, spectrum, and the motion --------------------------
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
axes[0].plot(xs, pluck(xs)*1000, "k--", lw=2, label="true triangle")
for N, col in ((1, "#e65100"), (3, "#2e7d32"), (10, "#1565c0")):
    axes[0].plot(xs, synth(N)*1000, lw=1.8, color=col, label=f"{N} harmonic(s)")
axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("y (mm)")
axes[0].set_title("Fourier synthesis converging"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

axes[1].stem(ns, np.abs(bg)*1000, basefmt=" ")
axes[1].set_xlabel("harmonic n"); axes[1].set_ylabel("$|b_n|$ (mm)")
axes[1].set_title("spectrum: every 3rd harmonic is absent"); axes[1].grid(alpha=.3)

for t, alpha in zip(np.linspace(0, 1/(2*f1), 5), (1, .75, .55, .4, .3)):
    yt = sum(b_given(n)*np.sin(n*np.pi*xs/L)*np.cos(2*np.pi*n*f1*t) for n in range(1, 60))
    axes[2].plot(xs, yt*1000, lw=1.6, color="#1565c0", alpha=alpha)
axes[2].set_xlabel("x (m)"); axes[2].set_ylabel("y (mm)")
axes[2].set_title("the string over half a period"); axes[2].grid(alpha=.3)
plt.suptitle("W13 P10 — plucked at L/3", y=1.04)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(v - 63.2456) < 1e-3 and abs(f1 - 15.8114) < 1e-3
assert abs(b_given(3)) < 1e-15 and abs(b_given(6)) < 1e-15
assert abs(b_given(1)*1000 - 15.7944) < 1e-3, f"b1 = {b_given(1)}"

# Final proof the corrected coefficients are the right ones: the reconstruction
# must reach the ACTUAL peak height h at the pluck point, not h/2.
peak_recon = max(sum(b_given(n)*np.sin(n*np.pi*xs/L) for n in range(1, 400)))
peak_printed = max(sum(b_printed(n)*np.sin(n*np.pi*xs/L) for n in range(1, 400)))
print(f"\n  reconstructed peak height with corrected b_n: {peak_recon*1000:.4f} mm "
      f"(true pluck height {h*1000:.1f} mm)")
print(f"  reconstructed peak height with printed   b_n: {peak_printed*1000:.4f} mm "
      f"<- only half the pluck")
assert abs(peak_recon - h) < 1e-4 and abs(peak_printed - h/2) < 1e-4

print(f"\n[OK] v = {v:.2f} m/s, f1 = {f1:.2f} Hz; CORRECTED b1 = {b_given(1)*1000:.3f} mm "
      f"(printed formula gives {b_printed(1)*1000:.3f} mm); harmonics 3, 6, 9, ... are silent.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_13.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
